# ForMoSA Advanced Plotting Tutorial

ForMoSA's `analysis.plot(results)` is convenient but opinionated: it runs corner, chains, radar, and best-fit plots all at once with default styling. For publication or presentation work, you typically want to (a) generate only the plot you need and (b) customize colors, fonts, labels, and legends.

This notebook covers all four plot types. You'll learn:

- Three techniques for loading nested sampling results
- How to call each plot individually
- The three layers of plotting config in ForMoSA: `PLOTS_CONFIG`, per-observation `plot_config`, and `MAIN_PLOT`
- How to post-process the returned matplotlib `Figure` / `Axes` for fine control
- A short matplotlib styling primer

**Prerequisites**: ForMoSA v2.0.0 (class version), and a completed nested sampling run (you'll need its `result_path`).

**Note on placeholders**: paths and parameter names below are placeholders. Replace them with values from your own fit.


---
## 1. Matplotlib styling primer

Before touching ForMoSA config, two matplotlib mechanisms are essential.

### 1.1 `rcParams` — global defaults

Set once at the top of the notebook. Everything plotted afterwards inherits these settings.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.size':        14,
    'font.family':      'serif',
    'font.serif':       ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',         # Computer Modern math (LaTeX-like, no LaTeX install required)
    'axes.linewidth':   1.2,
    'xtick.labelsize':  12,
    'ytick.labelsize':  12,
    'xtick.direction':  'in',
    'ytick.direction':  'in',
    'legend.fontsize':  11,
    'legend.frameon':   False,
    'figure.dpi':       100,
    'savefig.dpi':      300,
})

### 1.2 `plt.rc_context` — scoped overrides

If you want different fonts/styles for just one figure, use a context manager:

```python
with plt.rc_context({'font.size': 18, 'lines.linewidth': 2.0}):
    fig = plots.plot_corner()        # uses the overrides
# back to defaults outside the block
```

### 1.3 LaTeX rendering

Two paths:

- `mathtext.fontset='cm'` (above): math via `$...$` rendered in Computer Modern. No LaTeX install needed. Recommended default.
- `plt.rcParams['text.usetex'] = True`: uses your system's LaTeX. Slower, requires a working install, but lets you use arbitrary LaTeX (custom packages, `\textsc`, etc.) anywhere in a string.


---
## 2. Loading nested sampling results

ForMoSA stores fit output in two parallel files inside your `result_path`:

- `NS_results/results_<algo>.json` — the `NSResults` object (samples, weights, logl, logZ, etc.)
- `NS_params/NS_params.json` — the nested sampling configuration

Three loading patterns depending on what you want to plot.

### 2.1 Method A — `NSResults` from JSON (corner, chains, radar only)

Fastest. Sufficient for any plot that only uses posterior samples.

In [ ]:
import json
import logging
from ForMoSA.nested_sampling.results  import NSResults
from ForMoSA.nested_sampling.plotting import Plotting

# >>> REPLACE WITH YOUR PATH
results_json = 'PATH/TO/result_path/NS_results/results_pymultinest.json'

with open(results_json) as f:
    data = json.load(f)
results = NSResults.from_dict(data)

logger = logging.getLogger('plotting')
plots  = Plotting(results, logger=logger)

# Now you can call any of these:
# fig_corner          = plots.plot_corner()
# fig_chains, axs     = plots.plot_chains()
# fig_radar, ax_radar = plots.plot_radars()

### 2.2 Method B — Directly from PyMultiNest output

If you only have the raw output files (`RAW_stats.dat`, `RAW_ev.dat`, `RAW_.txt`):

In [ ]:
# >>> REPLACE
pymultinest_dir = 'PATH/TO/pymultinest/output_folder'
free_parameters = ['Teff', 'logg', 'M_H', 'C_O', 'r', 'd']   # adjust to your fit

results = NSResults.from_pymultinest(
    results_path    = pymultinest_dir,
    free_parameters = free_parameters,
)

### 2.3 Method C — Full `Analysis` (required for the best-fit plot)

The best-fit plot needs the **adapted model grid + observations + best-fit spectra**, not just samples. You must reconstruct the full pipeline state. Setting `fitted=True` skips the NS run and reconstructs `analysis.ns` and `analysis.ns_analysis` from disk.

In [ ]:
from ForMoSA.analysis import Analysis

# >>> REPLACE all config paths with yours
analysis = Analysis(
    # config_path        = ConfigPath(...),
    # config_parameters  = ConfigParameters(...),
    # config_inversion   = ConfigInversion(...),
    # config_NS          = ConfigNS(...),
    # config_adapt       = ConfigAdapt(...),
    adapted = True,    # adaptation already done on disk
    fitted  = True,    # NS already run; loads results from result_path
)

---
## 3. Best-fit plot

### 3.1 Calling only the best-fit plot

Bypass `analysis.plot()` (which runs all four plot types). Build `Plotting` and `NSAnalysis` yourself.

In [ ]:
from ForMoSA.nested_sampling.plotting   import Plotting
from ForMoSA.nested_sampling.ns_analysis import NSAnalysis

ns_analysis = NSAnalysis(analysis.ns, logger=analysis.logger)
plots       = Plotting(analysis.ns.results, analysis.logger)

fig, ax, ax_filt, axr, axr2 = plots.plot_fit(
    analysis.ns.restricted_observations,
    ns_analysis.best_fit,
)

Returned objects:

- `fig` — the matplotlib `Figure`
- `ax` — main spectrum panel
- `ax_filt` — photometric filter transmission panel (`None` if no photometry)
- `axr` — residuals panel (bottom)
- `axr2` — residual histogram (right of `axr`)

### 3.2 Customizing the best-fit line

Set the config **before** calling `plot_fit`:

In [ ]:
from ForMoSA.core.config import PLOTS_CONFIG

PLOTS_CONFIG.BestFitPlot.set_best_fit_plot_config(
    color_fit       = '#E8844A',
    color_residuals = '#2C2C2C',
    linewidth       = 1.5,
    zorder          = 200,
)

Available fields: `color_fit`, `color_residuals`, `linewidth`, `zorder`.

**No `alpha` field** in `BestFitPlotConfig` — for transparency on the best-fit line, edit post-hoc (see 3.5).

### 3.3 Customizing each observation

Every observation owns its own `plot_config`. Iterate `analysis.ns.restricted_observations` and set per-observation styling:

In [ ]:
# Example: assign distinct colors by observation name
custom_colors = {
    'SINFONI_K':   '#4C72B0',
    'HiRISE':      '#55A868',
    'photometry':  '#C44E52',
}

for obs in analysis.ns.restricted_observations:
    obs.plot_config.set_plot_config(
        color          = custom_colors.get(obs.name, obs.plot_config.color),
        linewidth      = 1.0,
        errorbar_alpha = 0.5,
    )

Fields available from `ObsPlotConfig`: `color`, `edgecolor`, `marker`, `markersize`, `linewidth`, `errorbar_fmt`, `errorbar_alpha`, `errorbar_capsize`, `zorder_data`, `zorder_error`, `label`.

### 3.4 Legend and figure-wide configuration

In [ ]:
from ForMoSA.core.config import MAIN_PLOT

MAIN_PLOT.figsize         = (20, 9)
MAIN_PLOT.legend_fontsize = 14
MAIN_PLOT.minor_ticks     = True
MAIN_PLOT.nb_minor_ticks  = 5

### 3.5 Post-hoc axis tweaks

ForMoSA's config doesn't expose everything (axis-label fontsize, tick fontsize, best-fit-line alpha). Edit the returned axes directly:

In [ ]:
# Re-render with current config
fig, ax, ax_filt, axr, axr2 = plots.plot_fit(
    analysis.ns.restricted_observations,
    ns_analysis.best_fit,
)

# Font sizes on main + residual panels
for a in [ax, axr]:
    a.tick_params(labelsize=14)
    a.xaxis.label.set_size(15)
    a.yaxis.label.set_size(15)

# Apply alpha to the best-fit line post-hoc
for line in ax.get_lines():
    if line.get_label() == 'Best fit':
        line.set_alpha(0.85)

# Redraw the legend with explicit styling
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, frameon=False, loc='upper right', fontsize=13)

### 3.6 Best-fit parameter values in the legend

`ns_analysis.ns.results.median_parameters` gives weighted posterior medians as `dict[str, float]`. For ±1σ uncertainties, use `_interval(sigma=1)` → `dict[str, (low, high)]`.

In [ ]:
medians   = ns_analysis.ns.results.median_parameters
intervals = ns_analysis.ns.results._interval(sigma=1)

parts = []
for k, med in medians.items():
    lo, hi = intervals[k]
    parts.append(f'{k}=${med:.2f}_{{-{med-lo:.2f}}}^{{+{hi-med:.2f}}}$')

new_label = 'Best fit:  ' + ',  '.join(parts)

for line in ax.get_lines():
    if line.get_label() == 'Best fit':
        line.set_label(new_label)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, frameon=False, loc='upper right', fontsize=11)

### 3.7 Plotting best-fit quantiles (1σ, 2σ bands)

`best_fit_interval(perc=...)` returns objects with `.wave` and `.flux` attributes.

> **Test on your setup**: I haven't verified whether `best_fit_interval` defaults to observation space or native-model space. The code in `analysis.plot()` only calls it when `plot_native_model=True`. If `lower.flux` has the wrong length for `fill_between` with your observation's wavelength grid, you'll need `ns_analysis.native_best_fit` instead, or compute the band manually by drawing samples (commented snippet below).

In [ ]:
lower_1, higher_1 = ns_analysis.best_fit_interval(perc=0.68)
lower_2, higher_2 = ns_analysis.best_fit_interval(perc=0.95)

ax.fill_between(lower_1.wave, lower_1.flux, higher_1.flux,
                color='grey', alpha=0.4, zorder=150, label='1$\\sigma$')
ax.fill_between(lower_2.wave, lower_2.flux, higher_2.flux,
                color='grey', alpha=0.2, zorder=140, label='2$\\sigma$')

ax.legend(*ax.get_legend_handles_labels(), frameon=False, loc='upper right', fontsize=11)

In [ ]:
# Fallback: manual quantile band by posterior sampling
# (use this if best_fit_interval doesn't give what you want)
#
# import numpy as np
#
# n_draws  = 500
# samples  = analysis.ns.results.samples
# weights  = analysis.ns.results.weights
# burn_in  = analysis.ns.results.burn_in
#
# # Weighted random draws
# w = weights[burn_in:] / weights[burn_in:].sum()
# idx = np.random.choice(len(w), size=n_draws, p=w, replace=True)
# draws = samples[burn_in:][idx]
#
# # Evaluate model at each draw (requires forward-model entry point — TBD)
# fluxes = []
# for params in draws:
#     model_flux = ...   # call your model evaluator here
#     fluxes.append(model_flux)
# fluxes = np.array(fluxes)
#
# lo1, hi1 = np.quantile(fluxes, [0.16, 0.84], axis=0)
# lo2, hi2 = np.quantile(fluxes, [0.025, 0.975], axis=0)
# ax.fill_between(wave_grid, lo1, hi1, color='grey', alpha=0.4)
# ax.fill_between(wave_grid, lo2, hi2, color='grey', alpha=0.2)

### 3.8 Interactive view

`%matplotlib widget` works in VS Code's Jupyter extension once `ipympl` is installed:

```bash
pip install ipympl
```

Then in the notebook:

In [ ]:
# Switch backends
%matplotlib widget        # interactive: pan, zoom, hover coordinates
# %matplotlib inline      # static PNG (default)

# Now re-render the plot to get the interactive figure
fig, ax, ax_filt, axr, axr2 = plots.plot_fit(
    analysis.ns.restricted_observations,
    ns_analysis.best_fit,
)

For richer interactivity (hover tooltips, downloadable HTML), convert to Plotly:

```python
import plotly.tools as tls
plotly_fig = tls.mpl_to_plotly(fig)
plotly_fig.show()
```

Plotly's matplotlib converter has limitations (some artist types don't translate well), so check the result visually.

### 3.9 Plotting a model at custom parameter values

**Deferred.** Generating a forward-model spectrum at arbitrary `(Teff, logg, [M/H], ...)` requires tracing the v2.0.0 grid-interpolation API (`SubGrid` interpolation + `ObservedModel` evaluation). I'll add this section in a follow-up once I've worked through the source.


---
## 4. Corner plot

`Plotting.plot_corner()` is a thin wrapper around the [`corner`](https://corner.readthedocs.io) library. The dataclass `PLOTS_CONFIG.CornerPlot` exposes most of `corner.corner`'s arguments.

### 4.1 Colors, contours, fills

In [ ]:
from ForMoSA.core.config import PLOTS_CONFIG

PLOTS_CONFIG.CornerPlot.set_corner_plot_config(
    color         = '#2E5C8A',
    fill_contours = True,
    plot_density  = True,
    plot_contours = True,
    smooth        = 1.2,         # Gaussian smoothing in pixels
    bins          = 60,
    # 1σ / 2σ / 3σ levels for a 2D Gaussian (corner's convention)
    levels        = [0.3935, 0.8647, 0.9889],
    hist_kwargs   = dict(color='#2E5C8A', histtype='stepfilled',
                         alpha=0.6, edgecolor='#143B66', linewidth=0.8),
    contour_kwargs= dict(colors='#2E5C8A', linewidths=0.8),
)

fig_corner = plots.plot_corner()

### 4.2 Fonts and label sizes

In [ ]:
PLOTS_CONFIG.CornerPlot.set_corner_plot_config(
    title_kwargs = dict(fontsize=15),
    label_kwargs = dict(fontsize=15),
    title_fmt    = ' .2f',
    max_n_ticks  = 4,
)

fig_corner = plots.plot_corner()

### 4.3 Replacing parameter labels with custom names

Corner labels come from `results.free_parameters`. Override post-hoc:

In [ ]:
import numpy as np

n = len(results.free_parameters)

# >>> REPLACE with your own LaTeX labels in the same order as free_parameters
custom_labels = [
    r'$T_{\mathrm{eff}}$ (K)',
    r'$\log g$',
    r'[M/H]',
    r'C/O',
    r'$R$ (R$_\mathrm{J}$)',
    r'$d$ (pc)',
]

axes = np.array(fig_corner.axes).reshape((n, n))

# Bottom row: x-labels
for i in range(n):
    axes[-1, i].set_xlabel(custom_labels[i], fontsize=15)

# Leftmost column: y-labels (skip the (0,0) panel which is a 1-D histogram)
for i in range(1, n):
    axes[i, 0].set_ylabel(custom_labels[i], fontsize=15)

fig_corner

### 4.4 Quantiles shown on the diagonal

The dashed lines on the diagonal show the quantiles you specify.

In [ ]:
PLOTS_CONFIG.CornerPlot.set_corner_plot_config(
    quantiles  = (0.025, 0.5, 0.975),   # show 95% range
    show_titles= True,
)

fig_corner = plots.plot_corner()

### 4.5 Zooming into specific panels

Corner uses the data range, weighted. If you need to override the range on a particular panel (e.g. clip a long posterior tail):

In [ ]:
axes = np.array(fig_corner.axes).reshape((n, n))

# Example: clip Teff range on the (0,0) histogram and on every panel that uses Teff
# axes[0, 0].set_xlim(1000, 2000)
# for i in range(1, n):
#     axes[i, 0].set_xlim(1000, 2000)     # left column = Teff x-axis
#     axes[0, i].set_ylim(0, None)        # top row = Teff y-axis (only if symmetric)

### 4.6 Tick density and font

If `max_n_ticks` in the config isn't enough control, set per-axis after the fact:

In [ ]:
from matplotlib.ticker import MaxNLocator

for a in fig_corner.axes:
    a.tick_params(labelsize=11)
    a.xaxis.set_major_locator(MaxNLocator(4))
    a.yaxis.set_major_locator(MaxNLocator(4))

fig_corner

---
## 5. Radar plot

The radar plot shows the posterior on each parameter as a normalized polar axis with a quantile band.

> Heads up: the `RadarPlotConfig` dataclass has a typo in the source — the field is spelled `fontisze_ticks`, not `fontsize_ticks`. Use the misspelled name.

In [ ]:
PLOTS_CONFIG.RadarPlot.set_radar_plot_config(
    color_radar        = '#7B3F8F',
    color_uncertainty  = '#B58CC2',
    color_quantiles    = '#7B3F8F',
    alpha_fill         = 0.4,
    linewidth          = 2.0,
    fontsize_names     = 13,
    fontisze_ticks     = 11,         # <-- note the typo
    color_ticks        = '#24292E',
    quantiles          = (0.16, 0.84),
    size_quantiles     = 80,
    lw_quantiles       = 2.0,
)

fig_radar, ax_radar = plots.plot_radars()

### 5.1 Replacing axis labels

In [ ]:
ax_radar.set_xticklabels(custom_labels, fontsize=13)
fig_radar

### 5.2 Adding annotations / text

In [ ]:
ax_radar.text(
    0.5, -0.12,
    'Shaded band: 16–84% posterior quantiles',
    transform=ax_radar.transAxes,
    ha='center', va='top',
    fontsize=11, color='grey',
)
fig_radar

---
## 6. Chains plot

The chains plot shows the raw NS samples in order (one panel per parameter), with the burn-in cutoff, importance weights (twinx), and best value overlaid.

In [ ]:
PLOTS_CONFIG.ChainsPlot.set_chains_plot_config(
    color_chains         = '#5E3C99',
    alpha_chains         = 0.6,

    # Burn-in marker
    color_plot_burn_in   = '#E66101',
    fontsize_burn_in     = 13,
    linestyle_burn_in    = '--',

    # Importance weights overlay (twinx)
    show_weights         = True,
    color_plot_weights   = '#1F1F1F',
    alpha_weights        = 0.35,
    fontsize_weights     = 13,

    # Best-value horizontal line
    plot_best_value      = True,
    color_best_value     = 'black',
    linestyle_best_value = '-.',
)

fig_chains, axs = plots.plot_chains()

### 6.1 Replacing y-labels with custom names

In [ ]:
for a, name in zip(axs, custom_labels):
    a.set_ylabel(name, fontsize=13)
    a.tick_params(labelsize=11)

fig_chains.tight_layout()
fig_chains

---
## 7. Saving figures

```python
fig         .savefig('best_fit_custom.pdf', dpi=300, bbox_inches='tight')
fig_corner  .savefig('corner.pdf',          dpi=300, bbox_inches='tight')
fig_radar   .savefig('radar.pdf',           dpi=300, bbox_inches='tight')
fig_chains  .savefig('chains.pdf',          dpi=300, bbox_inches='tight')
```

For raster output (PNG), `dpi=300` is the minimum for print. For vector formats (PDF, SVG), `dpi` only affects raster fall-backs (e.g. very dense scatter plots).

For journal submission, use PDF when possible — most journals prefer vector. If your figure has very dense imshow-based content, export as PNG at `dpi=600` and embed.
